In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

# import torch
# seed = 42
# torch.cuda.manual_seed_all(seed)

model_id = 'model'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).cuda()

streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=False, skip_special_tokens=False)
prompt = """\
Artificial intelligence is"""

_ = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(model.device),
    streamer=streamer,
    max_length=100,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
)
# print(tokenizer.decode(_[0]))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

model_id = "./model/base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).cuda()
streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=False, skip_special_tokens=False)

prompt = """
Question: What is the capital of Egypt?
Answer:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
_ = model.generate(
    **inputs,
    streamer=streamer,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
)


### logits for the NEXT token



In [ ]:
import torch

# tokenize
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# logits for the NEXT token
next_token_logits = logits[0, -1]

# probabilities
probs = torch.softmax(next_token_logits, dim=-1)

# top 20
topk = torch.topk(probs, k=20)

ids = topk.indices.tolist()
probs = topk.values.tolist()

for id, p in zip(ids, probs):
    print(f"{tokenizer.decode(id)}  {p*100:.2f}%")


### Assistant

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

model_id = 'assistant'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).cuda()

streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=False, skip_special_tokens=False)

In [ ]:
# import torch
# seed = 42
# torch.cuda.manual_seed_all(seed)
# Who invented Calculus?

messages = [
  {"role": "system", "content": "You are a helpful assistant."},
  {"role": "user", "content":"""What is the capital of France?"""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
_ = model.generate(
    **tokenizer(prompt, return_tensors="pt").to(model.device),
    streamer=streamer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.5,
    top_p=0.95,
    repetition_penalty=1.1,
)

### Classification

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_id = "./classifier"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id).cuda()

text = "I really love this movie"

inputs = tokenizer(text, truncation=True, max_length=64, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_id = probs.argmax(dim=-1).item()

print("pred_id:", pred_id)
print("label:", model.config.id2label[pred_id])
print("probs:", probs[0].tolist())

In [ ]:
from transformers import pipeline
clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,   # cuda:0
)

tests = [
    "Great, another update that broke everything.",
    "Not bad at all.",
    "I expected better.",
    "It's not the best, but it's far from terrible.",
    "Wow, just wow.",
    "Thanks for nothing.",
    "The movie was long.",
    "I guess it was fine."
]

for output in clf(tests):
    print(f"Label: {output['label']} -> Score: {output['score']:.4f}")